# Scraping Data UNESA untuk Program Studi S1 Sains Data

## Overview
Notebook ini digunakan untuk melakukan web scraping data akademik dari sistem informasi UNESA (Universitas Negeri Surabaya) untuk Program Studi S1 Sains Data. 

## Data yang di-scrape:
1. **RPS (Rencana Pembelajaran Semester)** - File PDF berisi rencana pembelajaran
2. **Jadwal Kuliah** - Jadwal mata kuliah per kelas dan hari
3. **Detail Mata Kuliah** - Informasi lengkap mata kuliah termasuk CPMK dan aktivitas

## Output yang dihasilkan:
- **PDF Files**: RPS mata kuliah disimpan dalam folder `rps_pdfs/`
- **CSV Files**: Data tabular disimpan dalam folder `file_tabulars/`
  - `jadwal_kuliah_s1_sains_data.csv` - Jadwal kuliah
  - `perkuliahan_s1_sains_data.csv` - Detail mata kuliah

## Prasyarat:
- Akun Google yang terdaftar untuk SSO UNESA
- File `.env` berisi kredensial login
- Koneksi internet yang stabil

---

In [1]:
# Uncomment lines below jika ingin install via notebook
# !pip install selenium webdriver-manager
# !pip install requests beautifulsoup4 lxml
# !pip install python-dotenv fake-useragent undetected-chromedriver
# !pip install pyotp qrcode[pil] pandas

## 1. Instalasi dan Import Libraries

### Instalasi Dependencies
Jalankan perintah berikut di terminal untuk menginstall semua library yang dibutuhkan:

```bash
pip install selenium webdriver-manager
pip install requests beautifulsoup4 lxml
pip install python-dotenv fake-useragent undetected-chromedriver
pip install pyotp qrcode[pil] pandas
```

Atau jika menggunakan pip, uncomment dan jalankan cell di bawah ini:

### Import Libraries
Import semua library yang diperlukan untuk web scraping dan pemrosesan data:

In [2]:
# Standard Library Imports
import os
import time
from pathlib import Path

# HTTP and Web Processing  
import requests
from bs4 import BeautifulSoup

# Data Processing
import pandas as pd
import re

print("📦 All necessary libraries imported successfully!")
print("🔧 Setup completed - ready for scraping!")

📦 All necessary libraries imported successfully!
🔧 Setup completed - ready for scraping!


## 2. Konfigurasi Environment dan Setup

### Setup Direktori Output
Membuat folder untuk menyimpan hasil scraping:

In [3]:
# Setup direktori untuk menyimpan hasil scraping
ROOT_DIR = Path().cwd()

# Direktori untuk file CSV dan data tabular
SAVE_DIR_TABULARS = ROOT_DIR / "file_tabulars"
SAVE_DIR_TABULARS.mkdir(parents=True, exist_ok=True)

# Direktori untuk file PDF RPS
SAVE_DIR_PDFS = ROOT_DIR / "rps_pdfs" 
SAVE_DIR_PDFS.mkdir(parents=True, exist_ok=True)

print(f"📂 Setup direktori completed:")
print(f"   📊 Data tabular: {SAVE_DIR_TABULARS}")
print(f"   📄 PDF files: {SAVE_DIR_PDFS}")

📂 Setup direktori completed:
   📊 Data tabular: C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars
   📄 PDF files: C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\rps_pdfs


## 3. Scraping RPS (Rencana Pembelajaran Semester) - File PDF

### Overview
Bagian ini melakukan scraping untuk mengunduh file PDF RPS dari halaman kurikulum S1 Sains Data UNESA.

**Proses:**
1. Ambil daftar mata kuliah dari halaman kurikulum
2. Untuk setiap mata kuliah, buka halaman detail
3. Cari link download PDF RPS
4. Download dan simpan PDF dengan nama yang rapi

**Output:** File PDF RPS disimpan di folder `rps_pdfs/`

In [4]:
# Konfigurasi untuk scraping RPS PDF
BASE_URL = "https://sindig.unesa.ac.id"
CURRICULUM_URL = f"{BASE_URL}/kurikulum/s1-sains-data"
PDF_DIR = str(SAVE_DIR_PDFS)  # Gunakan direktori yang sudah di-setup

def title_case(text: str) -> str:
    """
    Convert text ke Title Case yang rapi
    
    Args:
        text (str): Text input
        
    Returns:
        str: Text dalam format Title Case
    """
    return " ".join(word.capitalize() for word in text.split())

def ensure_dir(path):
    """
    Pastikan direktori ada, buat jika belum ada
    
    Args:
        path (str): Path direktori
    """
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"📂 Created directory: {path}")

def fetch_course_list(session):
    """
    Ambil daftar mata kuliah dari halaman kurikulum
    
    Args:
        session: Requests session object
        
    Returns:
        list: List berisi dict {'name', 'slug'} untuk setiap mata kuliah
    """
    print(f"📡 Fetching course list from: {CURRICULUM_URL}")
    
    try:
        resp = session.get(CURRICULUM_URL)
        resp.raise_for_status()
        
        soup = BeautifulSoup(resp.text, "html.parser")
        courses = []
        
        # Cari semua link mata kuliah
        course_links = soup.select("table a[href^='/mk/s1-sains-data/']")
        print(f"📋 Found {len(course_links)} course links")
        
        for a in course_links:
            slug = a["href"].rstrip("/").split("/")[-1]
            name = title_case(a.get_text(strip=True).lower())
            courses.append({"name": name, "slug": slug})
        
        # Remove duplicates berdasarkan slug
        unique_courses = {course["slug"]: course for course in courses}
        final_courses = list(unique_courses.values())
        
        print(f"✅ Processed {len(final_courses)} unique courses")
        return final_courses
        
    except Exception as e:
        print(f"❌ Error fetching course list: {e}")
        return []

def download_rps(session, course):
    """
    Download file RPS untuk satu mata kuliah
    
    Args:
        session: Requests session object
        course (dict): Dict berisi info mata kuliah {'name', 'slug'}
        
    Returns:
        bool: True jika berhasil download, False jika gagal
    """
    detail_url = f"{BASE_URL}/mk/s1-sains-data/{course['slug']}"
    
    try:
        print(f"🔍 Checking RPS for: {course['name']}")
        
        # Ambil halaman detail mata kuliah
        resp = session.get(detail_url)
        resp.raise_for_status()
        
        soup = BeautifulSoup(resp.text, "html.parser")
        
        # Cari link PDF RPS
        pdf_link = soup.select_one("a[href$='.pdf']")
        
        if not pdf_link:
            print(f"⚠️  RPS tidak tersedia untuk {course['name']}")
            return False
        
        # Buat URL lengkap untuk PDF
        from urllib.parse import urljoin
        pdf_url = urljoin(BASE_URL, pdf_link["href"])
        
        # Download PDF
        print(f"📥 Downloading RPS: {course['name']}")
        pdf_response = session.get(pdf_url)
        
        # Validasi response
        if pdf_response.status_code != 200:
            print(f"❌ Failed to download (HTTP {pdf_response.status_code}): {course['name']}")
            return False
            
        if not pdf_response.headers.get("Content-Type", "").startswith("application/pdf"):
            print(f"⚠️  File bukan PDF untuk: {course['name']}")
            return False
        
        # Buat nama file yang aman
        filename = f"{course['name']}.pdf"
        safe_filename = re.sub(r"[\\/*?\"<>|:]", "_", filename)
        file_path = os.path.join(PDF_DIR, safe_filename)
        
        # Simpan file
        with open(file_path, "wb") as f:
            f.write(pdf_response.content)
        
        file_size = len(pdf_response.content) / 1024  # Size in KB
        print(f"✅ Downloaded: {course['name']} ({file_size:.1f} KB)")
        return True
        
    except Exception as e:
        print(f"❌ Error downloading RPS for {course['name']}: {e}")
        return False

def scrape_all_rps():
    """
    Fungsi utama untuk scraping semua RPS PDF
    
    Returns:
        tuple: (success_count, total_count)
    """
    print("🚀 Starting RPS PDF scraping...")
    print("="*50)
    
    # Pastikan direktori PDF exists
    ensure_dir(PDF_DIR)
    
    success_count = 0
    
    # Gunakan session untuk efisiensi
    with requests.Session() as session:
        # Set user agent
        session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        
        # Ambil daftar mata kuliah
        courses = fetch_course_list(session)
        
        if not courses:
            print("❌ No courses found or error occurred")
            return 0, 0
        
        print(f"\n📚 Processing {len(courses)} courses...")
        print("="*50)
        
        # Download RPS untuk setiap mata kuliah
        for i, course in enumerate(courses, 1):
            print(f"\n[{i}/{len(courses)}] {course['name']}")
            if download_rps(session, course):
                success_count += 1
            
            # Delay untuk menghindari overload server
            time.sleep(0.5)
    
    print("\n" + "="*50)
    print(f"🎯 RPS PDF Scraping completed!")
    print(f"✅ Successfully downloaded: {success_count}/{len(courses)} files")
    print(f"📂 Files saved in: {PDF_DIR}")
    
    return success_count, len(courses)

# Jalankan scraping RPS
if __name__ == "__main__":
    scrape_all_rps()

🚀 Starting RPS PDF scraping...
📡 Fetching course list from: https://sindig.unesa.ac.id/kurikulum/s1-sains-data
📋 Found 247 course links
✅ Processed 93 unique courses

📚 Processing 93 courses...

[1/93] Agama
🔍 Checking RPS for: Agama
📥 Downloading RPS: Agama
✅ Downloaded: Agama (169.1 KB)

[2/93] Agama Budha
🔍 Checking RPS for: Agama Budha
📥 Downloading RPS: Agama Budha
✅ Downloaded: Agama Budha (278.4 KB)

[3/93] Agama Hindu
🔍 Checking RPS for: Agama Hindu
📥 Downloading RPS: Agama Hindu
✅ Downloaded: Agama Hindu (176.9 KB)

[4/93] Agama Islam
🔍 Checking RPS for: Agama Islam
📥 Downloading RPS: Agama Islam
✅ Downloaded: Agama Islam (299.6 KB)

[5/93] Agama Katholik
🔍 Checking RPS for: Agama Katholik
📥 Downloading RPS: Agama Katholik
✅ Downloaded: Agama Katholik (167.2 KB)

[6/93] Agama Khonghucu
🔍 Checking RPS for: Agama Khonghucu
📥 Downloading RPS: Agama Khonghucu
✅ Downloaded: Agama Khonghucu (133.6 KB)

[7/93] Agama Protestan
🔍 Checking RPS for: Agama Protestan
📥 Downloading RPS: Aga

## 4. Scraping Jadwal Kuliah S1 Sains Data

### Overview
Melakukan scraping jadwal kuliah untuk semua kelas S1 Sains Data dari halaman jadwal UNESA.

**Strategi Scraping:**
1. **Method 1**: Line-by-line parsing - parsing teks per baris untuk mendeteksi pola
2. **Method 2**: Regex pattern matching - menggunakan regular expression untuk ekstraksi data

**Output:** File `jadwal_kuliah_s1_sains_data.csv` berisi kolom:
- `kelas`: Nama kelas (contoh: Kelas A, Kelas B)  
- `hari`: Hari kuliah
- `jam`: Jam kuliah (format HH:MM)
- `mata_kuliah`: Nama mata kuliah

In [5]:
# URL untuk jadwal kuliah S1 Sains Data
SCHEDULE_URL = "https://sindig.unesa.ac.id/jadwal-kuliah/20251/s1-sains-data"

def scrape_class_schedule():
    print("HTML Table parsing...")
    try:
        session = requests.Session()
        session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        })
        response = session.get(SCHEDULE_URL)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        schedule_data = []

        tables = soup.find_all("table")
        print(f"   🔍 Found {len(tables)} tables")

        for table_idx, table in enumerate(tables):
            class_name = f"Unknown Class {table_idx+1}"
            
            # 1) Cari nama kelas dari h3.title dalam div.rbt-card parent
            rbt_card = table.find_parent("div", class_=lambda x: x and "rbt-card" in x)
            if rbt_card:
                title_h3 = rbt_card.find("h3", class_="title")
                if title_h3:
                    class_name = title_h3.get_text(strip=True)
                    print(f"   📌 Found class name in h3.title: {class_name}")
            
            # 2) Fallback: Coba ambil dari div.cart-table
            if class_name.startswith("Unknown"):
                cart = table.find_parent("div", class_="cart-table")
                if cart:
                    text = cart.get_text(" ", strip=True)
                    m = re.search(r'(Kelas\s*\w+)', text, re.IGNORECASE)
                    if m:
                        class_name = m.group(1)
                        print(f"   📌 Found class name in cart-table: {class_name}")

            # 3) Fallback lebih jauh: Cari di parent elements lain
            if class_name.startswith("Unknown"):
                prev_elements = []
                current = table.parent
                while current and len(prev_elements) < 10:
                    txt = current.get_text(strip=True)
                    if txt and re.search(r'kelas\s+\w+', txt, re.IGNORECASE) and len(txt)<50:
                        prev_elements.append(txt)
                    for sib in getattr(current, 'previous_siblings', []):
                        if hasattr(sib, 'get_text'):
                            txt2 = sib.get_text(strip=True)
                            if txt2 and re.search(r'kelas\s+\w+', txt2, re.IGNORECASE) and len(txt2)<50:
                                prev_elements.append(txt2)
                    current = current.parent
                for txt in prev_elements:
                    m2 = re.search(r'(Kelas\s*\w+)', txt, re.IGNORECASE)
                    if m2:
                        class_name = m2.group(1)
                        break

            print(f"   📌 Processing table {table_idx+1}: {class_name}")

            # Extract rows dari table
            for row in table.select("tbody tr"):
                cols = [td.get_text(strip=True) for td in row.find_all("td")]
                if len(cols) >= 3:
                    hari, jam, mata = cols[0], cols[1], cols[2]
                    days = ['Senin','Selasa','Rabu','Kamis','Jumat','Sabtu','Minggu']
                    if hari in days and re.match(r'^\d{1,2}[:.]\d{2}', jam):
                        jam = jam.replace('.',':')
                        if len(jam.split(':')[0])==1: 
                            jam = '0'+jam
                        if mata and len(mata)>2:
                            # Clean mata kuliah name (remove link tags)
                            mata_clean = re.sub(r'<[^>]+>', '', mata).strip()
                            schedule_data.append({
                                'kelas': class_name,
                                'hari': hari,
                                'jam': jam,
                                'mata_kuliah': mata_clean
                            })

        df = pd.DataFrame(schedule_data)
        print(f"   ✅ Method 1 found: {len(df)} schedule entries")
        return df

    except Exception as e:
        print(f"   ❌ Method 1 failed: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()


def scrape_schedule():
    """
    Fungsi utama untuk scraping jadwal dengan fallback methods
    
    Returns:
        pd.DataFrame: DataFrame final berisi jadwal kuliah
    """
    print("🚀 Starting schedule scraping...")
    print("="*50)
    
    # Try Method 1 first (HTML Table parsing)
    df = scrape_class_schedule()
    
    if not df.empty:
        # Data cleaning
        print("🧹 Cleaning data...")
        
        # Remove duplicates
        initial_count = len(df)
        df = df.drop_duplicates()
        print(f"   🗑️  Removed {initial_count - len(df)} duplicates")
        
        # Remove empty courses
        df = df[df['mata_kuliah'].str.strip() != '']
        df = df[df['mata_kuliah'].str.len() > 3]  # Minimum reasonable length
        
        # Clean mata kuliah names
        df['mata_kuliah'] = df['mata_kuliah'].str.replace(r'\s+', ' ', regex=True)
        df['mata_kuliah'] = df['mata_kuliah'].str.strip()
        
        # Clean class names
        df['kelas'] = df['kelas'].str.replace(r'\s+', ' ', regex=True)
        df['kelas'] = df['kelas'].str.strip()
        
        # Sort by kelas, hari, jam
        day_order = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
        df['hari_order'] = df['hari'].apply(lambda x: day_order.index(x) if x in day_order else 999)
        df = df.sort_values(['kelas', 'hari_order', 'jam']).drop('hari_order', axis=1)
        
        # Save to CSV
        output_file = SAVE_DIR_TABULARS / "jadwal_kuliah_s1_sains_data.csv"
        df.to_csv(output_file, index=False)
        
        print(f"✅ Schedule scraping completed!")
        print(f"📊 Total entries: {len(df)}")
        print(f"📝 Unique classes: {df['kelas'].nunique()}")
        print(f"📚 Unique courses: {df['mata_kuliah'].nunique()}")
        print(f"💾 Saved to: {output_file}")
        
        # Show summary per class
        print(f"\n📋 Summary per class:")
        class_summary = df.groupby('kelas').size().sort_index()
        for kelas, count in class_summary.items():
            print(f"   {kelas}: {count} jadwal")
        
    else:
        print("❌ No schedule data could be scraped")
    
    return df

# Jalankan scraping jadwal
if __name__ == "__main__":
    schedule_df = scrape_schedule()

🚀 Starting schedule scraping...
HTML Table parsing...
   🔍 Found 20 tables
   📌 Found class name in cart-table: Kelas 2025A
   📌 Processing table 1: Kelas 2025A
   📌 Found class name in cart-table: Kelas 2025B
   📌 Processing table 2: Kelas 2025B
   📌 Found class name in cart-table: Kelas 2025C
   📌 Processing table 3: Kelas 2025C
   📌 Found class name in cart-table: Kelas 2025D
   📌 Processing table 4: Kelas 2025D
   📌 Found class name in cart-table: Kelas 2025E
   📌 Processing table 5: Kelas 2025E
   📌 Found class name in cart-table: Kelas 2025F
   📌 Processing table 6: Kelas 2025F
   📌 Found class name in cart-table: Kelas 2025G
   📌 Processing table 7: Kelas 2025G
   📌 Found class name in cart-table: Kelas 2024A
   📌 Processing table 8: Kelas 2024A
   📌 Found class name in cart-table: Kelas 2024B
   📌 Processing table 9: Kelas 2024B
   📌 Found class name in cart-table: Kelas 2024C
   📌 Processing table 10: Kelas 2024C
   📌 Found class name in cart-table: Kelas 2024D
   📌 Processing

## 5. Scraping Detail Mata Kuliah dan Aktivitas Pembelajaran

### Overview  
Melakukan scraping detail lengkap untuk semua mata kuliah S1 Sains Data termasuk:
- Informasi mata kuliah (nama, kelas, program studi, semester)
- Deskripsi mata kuliah
- CPMK (Capaian Pembelajaran Mata Kuliah) 
- Daftar dosen pengampu
- Aktivitas pembelajaran per pertemuan

**Strategi CPMK Extraction:**
1. **Method 1**: Cari CPMK dengan kode (C1), (C2), dll
2. **Method 2**: Fallback dengan pattern "Mampu", "Menguasai", "Dapat"

**Output Files:**
- `unesa_courses.csv` - Data mata kuliah lengkap
- `unesa_activities.csv` - Detail aktivitas per pertemuan

In [7]:
# Konfigurasi untuk scraping course details
BASE_URL = "https://sindig.unesa.ac.id"
COURSE_LIST_URL = f"{BASE_URL}/list-course/s1-sains-data"

def fetch_all_course_links():
    """
    Ambil semua link mata kuliah dari list-course dengan pagination
    
    Returns:
        list: List tuple berisi (url, title) untuk setiap mata kuliah
    """
    print("🔍 Fetching all course links...")
    
    links = []
    page = 1
    
    while True:
        url = f"{COURSE_LIST_URL}?p={page}&"
        print(f"   📄 Checking page {page}")
        
        try:
            resp = requests.get(url)
            
            # Jika 404, berarti tidak ada halaman lagi
            if resp.status_code == 404:
                print(f"   🛑 No more pages (404 on page {page})")
                break
                
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "html.parser")
            
            # Cari card mata kuliah
            course_cards = soup.select("h4.rbt-card-title a[href^='/course/']")
            
            if not course_cards:
                print(f"   🛑 No courses found on page {page}")
                break
            
            print(f"   ✅ Found {len(course_cards)} courses on page {page}")
            
            # Extract URL dan title
            for card in course_cards:
                from urllib.parse import urljoin
                full_url = urljoin(BASE_URL, card["href"])
                title = card.get_text(strip=True)
                links.append((full_url, title))
            
            page += 1
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            print(f"   ❌ Error on page {page}: {e}")
            break
    
    print(f"🎯 Total course links found: {len(links)}")
    return links

def extract_course_details(course_url, raw_title):
    """
    Extract detail lengkap dari satu mata kuliah
    
    Args:
        course_url (str): URL halaman mata kuliah
        raw_title (str): Judul mentah mata kuliah
        
    Returns:
        dict: course_record saja (tanpa activities list)
    """
    try:
        resp = requests.get(course_url)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        
        # Parse title dan kelas
        if "|" in raw_title:
            mata_kuliah, kelas = [part.strip() for part in raw_title.split("|", 1)]
        else:
            mata_kuliah, kelas = raw_title.strip(), ""
        
        print(f"       Processing: {mata_kuliah} | {kelas}")
        
        # Extract metadata (Program Studi, Semester, Lectures)
        program_studi = semester = lectures_count = ""
        
        for li in soup.select("ul.has-show-more-inner-content.rbt-course-details-list-wrapper li"):
            spans = li.select("span")
            if len(spans) == 2:
                label = spans[0].get_text(strip=True)
                value = spans[1].get_text(strip=True)
                
                if label == "Program Studi":
                    program_studi = value
                elif label == "Semester":
                    semester = value
                elif label == "Lectures":
                    lectures_count = value
        
        # Extract description
        desc_element = soup.select_one("div.rbt-course-feature-inner.has-show-more-inner-content > p")
        description = desc_element.get_text(strip=True) if desc_element else ""
        
        # Extract CPMK dengan robust method
        cpmk_items = []
        
        # Method 1: Cari CPMK dengan kode (C1), (C2), dll
        for block in soup.select("div.row.g-5.mb--30"):
            for li in block.select("ul.rbt-list-style-1 li"):
                text = li.get_text(" ", strip=True).replace("\\", "")
                if re.search(r"\(C\d+\)", text):
                    cpmk_items.append(text)
        
        # Method 2: Fallback jika tidak ada coded CPMK
        if not cpmk_items:
            header = soup.find(lambda tag: tag.name == "h4" and "CPMK" in tag.get_text())
            if header and header.parent:
                for ul in header.parent.find_all("ul", class_="rbt-list-style-1"):
                    for li in ul.find_all("li"):
                        text = li.get_text(" ", strip=True).replace("\\", "")
                        # Pattern alternatif untuk CPMK
                        if (text.startswith(("Mampu", "Menguasai", "Dapat")) or len(text) > 20):
                            cpmk_items.append(text)
        
        cpmk_combined = " | ".join(cpmk_items)
        print(f"       CPMK found: {len(cpmk_items)} items")
        
        # Extract lecturers
        lecturers = []
        for lecturer_link in soup.select("div.author-info h5.title a"):
            name = lecturer_link.get_text(strip=True).title()
            if name:
                lecturers.append(name)
        
        lecturer_1 = lecturers[0] if len(lecturers) > 0 else ""
        lecturer_2 = lecturers[1] if len(lecturers) > 1 else ""
        
        print(f"       Lecturers: {', '.join(lecturers[:2])}")
        
        # Build course record dengan kolom pertemuan dan date kosong dulu
        course_record = {
            "course_url": course_url,
            "mata_kuliah": mata_kuliah,
            "kelas": kelas,
            "program_studi": program_studi,
            "semester": semester,
            "lectures_count": lectures_count,
            "description": description,
            "cpmk": cpmk_combined,
            "pengampu_1": lecturer_1,
            "pengampu_2": lecturer_2
        }
        
        # Add individual meeting columns (pertemuan_1 to pertemuan_16)
        for i in range(1, 17):
            course_record[f"pertemuan_{i}"] = ""
            course_record[f"date_{i}"] = ""
        
        # Extract activities dan langsung masukkan ke course_record
        activity_ul = soup.select_one(
            "div.course-content.rbt-shadow-box.coursecontent-wrapper "
            "div.rbt-course-feature-inner ul.rbt-list-style-1"
        )
        
        activities_count = 0
        if activity_ul:
            for li in activity_ul.find_all("li", recursive=False):
                table = li.find("table")
                if not table:
                    continue
                
                # Extract meeting number
                meeting_cell = table.select_one("td strong")
                meeting_num = ""
                if meeting_cell:
                    meeting_text = meeting_cell.get_text(strip=True)
                    meeting_num = meeting_text.split()[-1] if meeting_text else ""
                
                # Extract description dan date
                content_cells = table.find_all("td")
                if len(content_cells) > 1:
                    content_cell = content_cells[1]
                    
                    # Extract date DULU sebelum mengambil text
                    date_element = content_cell.find("date")
                    date_text = date_element.get_text(strip=True) if date_element else ""
                    
                    # Hapus elemen date dari DOM sebelum ambil description
                    if date_element:
                        date_element.decompose()
                    
                    # Ambil description SETELAH date element dihapus
                    description_text = content_cell.get_text(" ", strip=True)
                    
                    # Clean up description lebih lanjut
                    description_text = re.sub(r"\s*Date\s*$", "", description_text).strip()
                    description_text = re.sub(r"\s+", " ", description_text).strip()
                    
                    # Fill meeting data jika meeting number valid
                    if meeting_num.isdigit() and 1 <= int(meeting_num) <= 16:
                        course_record[f"pertemuan_{meeting_num}"] = description_text
                        course_record[f"date_{meeting_num}"] = date_text
                        activities_count += 1
        
        course_record["total_activities"] = activities_count
        print(f"       Activities processed: {activities_count}")
        
        return course_record
        
    except Exception as e:
        print(f"   ❌ Error processing {course_url}: {e}")
        return None

def scrape_all_course_details():
    """
    Fungsi utama untuk scraping semua detail mata kuliah
    
    Returns:
        pd.DataFrame: courses_df saja
    """
    print("🚀 Starting comprehensive course details scraping...")
    print("="*60)
    
    # Fetch all course links
    course_links = fetch_all_course_links()
    
    if not course_links:
        print("❌ No course links found")
        return pd.DataFrame()
    
    course_records = []
    
    print(f"🔄 Processing {len(course_links)} courses...")
    print("="*60)
    
    # Process each course
    for index, (url, title) in enumerate(course_links, 1):
        print(f"[{index}/{len(course_links)}] {title}")
        
        course_record = extract_course_details(url, title)
        
        if course_record:
            course_records.append(course_record)
        
        # Rate limiting
        time.sleep(0.5)
    
    # Create DataFrame
    courses_df = pd.DataFrame(course_records)
    
    print("\n" + "="*60)
    print("📊 Scraping Results:")
    print(f"   📚 Courses processed: {len(courses_df)}")
    
    if not courses_df.empty:
        # Save courses data
        courses_file = SAVE_DIR_TABULARS / "perkuliahan_sains_data.csv"
        courses_df.to_csv(courses_file, index=False)
        print(f"   💾 Courses saved to: {courses_file}")
        
        # Show courses summary
        print(f"   📋 Courses Summary:")
        print(f"     • Unique programs: {courses_df['program_studi'].nunique()}")
        print(f"     • Unique semesters: {courses_df['semester'].nunique()}")
        print(f"     • Courses with lecturers: {courses_df[courses_df['lecturer_1'] != ''].shape[0]}")
        print(f"     • Courses with CPMK: {courses_df[courses_df['cpmk'] != ''].shape[0]}")
        print(f"     • Total activities captured: {courses_df['total_activities'].sum()}")

    print("✅ Course details scraping completed!")
    return courses_df

# Jalankan scraping course details
if __name__ == "__main__":
    courses_df = scrape_all_course_details()

🚀 Starting comprehensive course details scraping...
🔍 Fetching all course links...
   📄 Checking page 1
   ✅ Found 6 courses on page 1
   📄 Checking page 2
   ✅ Found 6 courses on page 2
   📄 Checking page 3
   ✅ Found 6 courses on page 3
   📄 Checking page 4
   ✅ Found 6 courses on page 4
   📄 Checking page 5
   ✅ Found 6 courses on page 5
   📄 Checking page 6
   ✅ Found 6 courses on page 6
   📄 Checking page 7
   ✅ Found 6 courses on page 7
   📄 Checking page 8
   ✅ Found 6 courses on page 8
   📄 Checking page 9
   ✅ Found 6 courses on page 9
   📄 Checking page 10
   ✅ Found 6 courses on page 10
   📄 Checking page 11
   ✅ Found 6 courses on page 11
   📄 Checking page 12
   ✅ Found 6 courses on page 12
   📄 Checking page 13
   ✅ Found 6 courses on page 13
   📄 Checking page 14
   ✅ Found 6 courses on page 14
   📄 Checking page 15
   ✅ Found 6 courses on page 15
   📄 Checking page 16
   ✅ Found 6 courses on page 16
   📄 Checking page 17
   ✅ Found 6 courses on page 17
   📄 Checking pag

KeyError: 'lecturer_1'

## 6. Menjalankan Semua Proses Scraping

### Eksekusi Complete Pipeline
Jalankan cell di bawah untuk menjalankan semua proses scraping sekaligus:

In [9]:
def run_complete_scraping():
    """
    Menjalankan semua proses scraping secara berurutan
    """
    print("🚀 STARTING COMPLETE UNESA SCRAPING PIPELINE")
    print("="*70)
    
    start_time = time.time()
    
    try:
        # 1. Scraping RPS PDF
        print("\\n📄 PHASE 1: RPS PDF Scraping")
        print("-" * 40)
        rps_success, rps_total = scrape_all_rps()
        
        # 2. Scraping Jadwal Kuliah  
        print("\\n📅 PHASE 2: Schedule Scraping")
        print("-" * 40)
        schedule_df = scrape_schedule_combined()
        
        # 3. Scraping Course Details
        print("\\n📚 PHASE 3: Course Details Scraping") 
        print("-" * 40)
        courses_df, activities_df = scrape_all_course_details()
        
        # Summary
        elapsed_time = time.time() - start_time
        
        print("\\n" + "="*70)
        print("🎯 SCRAPING PIPELINE COMPLETED!")
        print("="*70)
        print(f"⏱️  Total execution time: {elapsed_time/60:.1f} minutes")
        print("\\n📊 Final Results Summary:")
        print(f"   📄 RPS PDFs: {rps_success}/{rps_total} downloaded")
        print(f"   📅 Schedule entries: {len(schedule_df) if not schedule_df.empty else 0}")
        print(f"   📚 Courses processed: {len(courses_df) if not courses_df.empty else 0}")
        print(f"   📋 Activities found: {len(activities_df) if not activities_df.empty else 0}")
        
        print(f"\\n📂 Output Files:")
        print(f"   📄 PDF files: {SAVE_DIR_PDFS}/")
        print(f"   📊 CSV files: {SAVE_DIR_TABULARS}/")
        print(f"      - jadwal_kuliah_s1_sains_data.csv")
        print(f"      - unesa_courses.csv") 
        print(f"      - unesa_activities.csv")
        
        return True
        
    except Exception as e:
        print(f"\\n❌ Pipeline failed: {e}")
        return False

# Uncomment line di bawah untuk menjalankan complete pipeline
run_complete_scraping()

🚀 STARTING COMPLETE UNESA SCRAPING PIPELINE
\n📄 PHASE 1: RPS PDF Scraping
----------------------------------------
🚀 Starting RPS PDF scraping...
📡 Fetching course list from: https://sindig.unesa.ac.id/kurikulum/s1-sains-data
📋 Found 247 course links
✅ Processed 93 unique courses

📚 Processing 93 courses...

[1/93] Agama
🔍 Checking RPS for: Agama
📥 Downloading RPS: Agama
✅ Downloaded: Agama (169.1 KB)

[2/93] Agama Budha
🔍 Checking RPS for: Agama Budha
📥 Downloading RPS: Agama Budha
✅ Downloaded: Agama Budha (278.5 KB)

[3/93] Agama Hindu
🔍 Checking RPS for: Agama Hindu
📥 Downloading RPS: Agama Hindu
✅ Downloaded: Agama Hindu (176.8 KB)

[4/93] Agama Islam
🔍 Checking RPS for: Agama Islam
📥 Downloading RPS: Agama Islam
✅ Downloaded: Agama Islam (299.6 KB)

[5/93] Agama Katholik
🔍 Checking RPS for: Agama Katholik
📥 Downloading RPS: Agama Katholik
✅ Downloaded: Agama Katholik (167.2 KB)

[6/93] Agama Khonghucu
🔍 Checking RPS for: Agama Khonghucu
📥 Downloading RPS: Agama Khonghucu
✅ Downlo

KeyboardInterrupt: 